# VarChat — model dosyası üret (GGUF)

Drive'daki eğitilmiş adaptörü (`varchat-lora-v2/`) taban modele (Qwen2.5-7B-Instruct) kalıcı olarak ekler, Ollama'nın
çalıştırabileceği tek dosyaya çevirir ve 4-bit'e sıkıştırır (Q4_K_M). Sonuç: Drive'da **`varchat-gguf/varchat-v2-Q4_K_M.gguf`** (~4,7 GB).
Ölçüm yapmaz. Adımlar `colab_olcum.ipynb`'nin 6. bölümüyle aynıdır; ölçülen modelin aynısı üretilir.

**Önce:** Drive'da `varchat-lora-v2` klasörü olsun ve içinde `adapter_model.safetensors` + `adapter_config.json` bulunsun.

**Kullanım:** *Runtime → Change runtime type → **CPU** + **High-RAM*** (GPU gerekmez, kredi az harcar) → *Run all*.
Birinci hücre RAM ya da disk yetmiyor derse **L4** seç ve baştan *Run all* yap. Süre ~40-60 dk.

## Ayar

In [ ]:
VARCHAT = "varchat-v2"
LORA_DIR = '/content/drive/MyDrive/varchat-lora-v2'
GGUF_DRIVE = '/content/drive/MyDrive/varchat-gguf'

## 1) Bellek, disk ve Drive

In [ ]:
import os, shutil
RAM_GB = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
DISK_GB = shutil.disk_usage('/content').free / 1e9
print(f"RAM: {RAM_GB:.0f} GB | boş disk: {DISK_GB:.0f} GB")
assert RAM_GB >= 30, "RAM yetmiyor: Runtime → Change runtime type → High-RAM (ya da L4) seç, baştan Run all yap."
assert DISK_GB >= 50, "Disk yetmiyor: Runtime → Change runtime type → L4 seç, baştan Run all yap."

from google.colab import drive
drive.mount('/content/drive')
os.makedirs(GGUF_DRIVE, exist_ok=True)
for ad in ("adapter_model.safetensors", "adapter_config.json"):
    assert os.path.exists(f"{LORA_DIR}/{ad}"), f"Drive'da {LORA_DIR}/{ad} yok: önce adaptör dosyalarını Drive'a yükle."
bos = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
print(f"Drive boş yer: {bos:.1f} GB")
assert bos > 6, "Drive'da en az 6 GB boş yer olmalı (model dosyası ~4,7 GB)."

## 2) Araçlar (eğitimdekiyle aynı sürümler) + llama.cpp

In [ ]:
!pip install -q "transformers==5.17.0" "peft==0.21.0" "accelerate==1.15.0" sentencepiece "protobuf>=4.21,<5"
# Colab'ın eski torchao'su peft 0.21'in birleştirme yolunu bozuyor; bu iş için gerekmez
!pip uninstall -y -q torchao
!apt-get -qq install -y cmake libcurl4-openssl-dev > /dev/null
if not os.path.exists('/content/llama.cpp'):
    !git clone -q https://github.com/ggml-org/llama.cpp /content/llama.cpp
QUANTIZE = '/content/llama.cpp/build/bin/llama-quantize'
if not os.path.exists(QUANTIZE):
    print("llama-quantize derleniyor (~3-5 dk)...")
    !cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_SERVER=OFF > /content/cmake.log 2>&1 || tail -20 /content/cmake.log
    !cmake --build /content/llama.cpp/build --target llama-quantize -j 8 > /content/build.log 2>&1 || tail -30 /content/build.log
assert os.path.exists(QUANTIZE), "llama-quantize derlenemedi; /content/build.log dosyasına bak"
LLAMA_COMMIT = !git -C /content/llama.cpp log --oneline -1
print("llama.cpp:", LLAMA_COMMIT[0])

## 3) Taban modeli indir (~15 GB) ve adaptörü birleştir (CPU, ~10-15 dk)

In [ ]:
import gc, torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
TABAN_HF = snapshot_download("Qwen/Qwen2.5-7B-Instruct")
print("taban model:", TABAN_HF)
if not os.path.exists('/content/varchat-hf/config.json'):
    print("birleştiriliyor...")
    taban = AutoModelForCausalLM.from_pretrained(TABAN_HF, dtype=torch.bfloat16, device_map="cpu", low_cpu_mem_usage=True)
    birlesik = PeftModel.from_pretrained(taban, LORA_DIR).merge_and_unload()
    birlesik.save_pretrained('/content/varchat-hf', safe_serialization=True, max_shard_size="4GB")
    AutoTokenizer.from_pretrained(TABAN_HF).save_pretrained('/content/varchat-hf')
    if os.path.exists(f"{TABAN_HF}/generation_config.json"):
        shutil.copy(f"{TABAN_HF}/generation_config.json", '/content/varchat-hf/')
    del taban, birlesik
    gc.collect()
print("birleştirildi -> /content/varchat-hf")

## 4) GGUF'a çevir, Q4_K_M'e sıkıştır, Drive'a kaydet

In [ ]:
import subprocess, sys, json, datetime, hashlib

def donustur(python):
    return subprocess.run([python, "/content/llama.cpp/convert_hf_to_gguf.py", '/content/varchat-hf', "--outtype", "f16",
                           "--outfile", '/content/varchat-f16.gguf'], capture_output=True, text=True)

def ayri_ortam():
    # Dönüştürme betiği yeni transformers ile çalışmazsa: transformers<5 olan ayrı sanal ortam
    py = "/content/donustur-venv/bin/python"
    if not os.path.exists(py):
        !python -m venv /content/donustur-venv || (pip install -q virtualenv && python -m virtualenv -q /content/donustur-venv)
        !/content/donustur-venv/bin/pip install -q torch --index-url https://download.pytorch.org/whl/cpu
        !/content/donustur-venv/bin/pip install -q "transformers>=4.45,<5" sentencepiece "protobuf>=4.21,<5" numpy safetensors pyyaml tqdm requests
    return py

PYTHON = sys.executable
print("GGUF'a çevriliyor...")
r = donustur(PYTHON)
if r.returncode != 0:
    print("   bu ortamda olmadı, ayrı ortamda tekrar deneniyor:", r.stderr.strip().splitlines()[-1:])
    PYTHON = ayri_ortam()
    r = donustur(PYTHON)
if r.returncode != 0:
    print(r.stdout[-1500:], r.stderr[-3000:])
    raise SystemExit("GGUF dönüştürme başarısız (çıktıyı Claude'a gönder)")

YEREL = f"/content/{VARCHAT}-Q4_K_M.gguf"
print("Q4_K_M'e sıkıştırılıyor...")
r = subprocess.run([QUANTIZE, '/content/varchat-f16.gguf', YEREL, "Q4_K_M"], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-1500:], r.stderr[-1500:])
    raise SystemExit("Q4_K_M sıkıştırma başarısız (çıktıyı Claude'a gönder)")
os.remove('/content/varchat-f16.gguf')

h = hashlib.sha256()
with open(YEREL, "rb") as f:
    for parca in iter(lambda: f.read(1 << 24), b""):
        h.update(parca)
HEDEF = f"{GGUF_DRIVE}/{VARCHAT}-Q4_K_M.gguf"
shutil.copy(YEREL, HEDEF)
bilgi = {"tarih": datetime.datetime.now().isoformat(timespec="minutes"), "llama_cpp": LLAMA_COMMIT[0],
         "donusturme_python": PYTHON, "dosya": os.path.basename(HEDEF), "boyut": os.path.getsize(HEDEF), "sha256": h.hexdigest()}
json.dump(bilgi, open(f"{GGUF_DRIVE}/{VARCHAT}_gguf_bilgi.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(json.dumps(bilgi, ensure_ascii=False, indent=1))
print(f"\nHAZIR: Drive -> varchat-gguf/{VARCHAT}-Q4_K_M.gguf ({os.path.getsize(HEDEF) / 1e9:.2f} GB)")

## 5) Bilgisayarına al

Drive'da **`varchat-gguf/varchat-v2-Q4_K_M.gguf`** dosyasına sağ tıkla → *İndir* (klasörü değil, yalnız bu dosyayı).
"Virüs taraması yapılamıyor" uyarısında *Yine de indir* de. Dosyayı projede `modeller/` klasörüne koy.
Sonra *Runtime → Disconnect and delete runtime* ile oturumu kapat (kredi harcamasın).